## Data loading and configuration

In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime
import glob

# Import the aggregation function from ActiveStrategyFramework
from ActiveStrategyFramework import aggregate_price_data

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Imports successful!")

Imports successful!


## Load and Process Reference Pool (USDC/ETH)

In [2]:
# Configuration
DATA_DIR = './data'
POOL_DATA_DIR = os.path.join(DATA_DIR, 'centralized_prices', 'spot')
FREQUENCY = 'H'  # Hourly aggregation

# Get all pool CSV files (excluding the eth_usd reference)
pool_csv_files = [f for f in glob.glob(os.path.join(DATA_DIR, '*_bigquery.csv')) 
                  if 'usdc_weth' not in f.lower()]

print(f"Found {len(pool_csv_files)} pool data files:")
for file in pool_csv_files:
    print(f"  - {os.path.basename(file)}")

print(f"\nFound {len(pool_csv_files)} pool files to process")

Found 6 pool data files:
  - vita_weth_1_bigquery.csv
  - link_weth_03_bigquery.csv
  - wtao_weth_1_bigquery.csv
  - morpho_weth_03_bigquery.csv
  - rndr_weth_03_bigquery.csv
  - weth_inj_03_bigquery.csv

Found 6 pool files to process


In [5]:
# Load ETH/USD price data from Binance
eth_usd_file = os.path.join(POOL_DATA_DIR, 'eth_binance_5m.csv')

if os.path.exists(eth_usd_file):
    print(f"Loading ETH/USD price data from: {eth_usd_file}")
    eth_usd_data = pd.read_csv(eth_usd_file, index_col='datetime', parse_dates=True)
    
    # Ensure timezone consistency - convert to UTC if needed
    if eth_usd_data.index.tz is None:
        print("ETH/USD data is timezone-naive, converting to UTC...")
        eth_usd_data.index = eth_usd_data.index.tz_localize('UTC')
    else:
        print(f"ETH/USD data timezone: {eth_usd_data.index.tz}")
        # Convert to UTC if it's not already
        eth_usd_data.index = eth_usd_data.index.tz_convert('UTC')
    
    # Resample to hourly (same frequency as pool data)
    eth_usd_hourly = eth_usd_data['close'].resample('H').last().dropna()
    
    print(f"ETH/USD data shape: {eth_usd_data.shape}")
    print(f"ETH/USD hourly shape: {eth_usd_hourly.shape}")
    print(f"ETH/USD price range: ${eth_usd_hourly.min():.2f} - ${eth_usd_hourly.max():.2f}")
    print(f"ETH/USD date range: {eth_usd_hourly.index.min()} to {eth_usd_hourly.index.max()}")
    print(f"ETH/USD timezone: {eth_usd_hourly.index.tz}")
    
    # Show sample of ETH/USD prices
    print(f"\nSample ETH/USD prices:")
    print(eth_usd_hourly.head(10))
    
else:
    print(f"❌ ETH/USD price file not found: {eth_usd_file}")
    print("Please download ETH/USD data from Binance first using CCXTDataFetcher")

Loading ETH/USD price data from: ./data/centralized_prices/spot/eth_binance_5m.csv
ETH/USD data is timezone-naive, converting to UTC...
ETH/USD data shape: (131229, 6)
ETH/USD hourly shape: (10936,)
ETH/USD price range: $1418.80 - $4935.00
ETH/USD date range: 2024-05-31 22:00:00+00:00 to 2025-08-30 13:00:00+00:00
ETH/USD timezone: UTC

Sample ETH/USD prices:
datetime
2024-05-31 22:00:00+00:00    3770.38
2024-05-31 23:00:00+00:00    3762.29
2024-06-01 00:00:00+00:00    3764.15
2024-06-01 01:00:00+00:00    3766.79
2024-06-01 02:00:00+00:00    3777.79
2024-06-01 03:00:00+00:00    3778.98
2024-06-01 04:00:00+00:00    3784.35
2024-06-01 05:00:00+00:00    3789.79
2024-06-01 06:00:00+00:00    3788.84
2024-06-01 07:00:00+00:00    3773.00
Freq: h, Name: close, dtype: float64


/var/folders/qq/b2n1bnrj4dd7b2qz0_1_8gfm0000gn/T/ipykernel_46710/1142973441.py:18: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  eth_usd_hourly = eth_usd_data['close'].resample('H').last().dropna()


In [14]:
def process_pool_with_eth_usd(pool_file, eth_usd_prices):
    """
    Process pool data and calculate USD prices using ETH/USD reference
    """
    pool_name = os.path.basename(pool_file).replace('_bigquery.csv', '')
    print(f"\n{'='*60}")
    print(f"Processing pool: {pool_name}")
    print(f"{'='*60}")
    
    try:
        # Load pool data
        pool_data = pd.read_csv(pool_file, index_col='block_date', parse_dates=True)
        print(f"Raw pool data shape: {pool_data.shape}")
        print(f"Pool data timezone: {pool_data.index.tz}")
        
        # Ensure timezone consistency - convert to UTC if needed
        if pool_data.index.tz is None:
            print("Pool data is timezone-naive, converting to UTC...")
            pool_data.index = pool_data.index.tz_localize('UTC')
        else:
            print(f"Pool data timezone: {pool_data.index.tz}")
            # Convert to UTC if it's not already
            pool_data.index = pool_data.index.tz_convert('UTC')
        
        # Aggregate to hourly using ActiveStrategyFramework function
        print("Aggregating to hourly...")
        hourly_data = aggregate_price_data(pool_data, frequency='H')  # Use 'H' (uppercase) for hourly
        
        if hourly_data is None or hourly_data.empty:
            print(f"❌ Failed to aggregate data for {pool_name}")
            return None
        
        print(f"Hourly aggregated shape: {hourly_data.shape}")
        print(f"Hourly data timezone: {hourly_data.index.tz}")
        
        # Ensure ETH/USD prices have the same timezone
        eth_usd_aligned = eth_usd_prices.copy()
        if eth_usd_aligned.index.tz != hourly_data.index.tz:
            print("Aligning ETH/USD timezone with pool data...")
            eth_usd_aligned.index = eth_usd_aligned.index.tz_convert(hourly_data.index.tz)
        
        # Calculate USD prices using ETH/USD reference
        print("Calculating USD prices...")
        
        # Get the quotePrice from pool data
        if 'quotePrice' in hourly_data.columns:
            # Align the ETH/USD prices with the pool data timestamps
            # Use forward fill to handle any missing timestamps
            eth_prices_aligned = eth_usd_aligned.reindex(hourly_data.index, method='ffill')
            
            # Special handling for INJ pool (inverted)
            if 'inj' in pool_name.lower():
                print("INJ pool detected - using inverted calculation")
                print("quotePrice = INJ/ETH, so INJ/USD = (1/quotePrice) × ETH/USD")
                # For INJ pool: INJ/USD = (1/quotePrice) × ETH/USD
                token1_usd_price = (1 / hourly_data['quotePrice']) * eth_prices_aligned
            else:
                # Normal calculation for other pools
                print("Normal pool calculation")
                print("quotePrice = token1/token0, so token1/USD = quotePrice × ETH/USD")
                token1_usd_price = hourly_data['quotePrice'] * eth_prices_aligned
            
            # Add USD price columns
            hourly_data['token1_usd_price'] = token1_usd_price
            hourly_data['eth_usd_price'] = eth_prices_aligned
            
            print(f"USD price calculation complete")
            print(f"Token1 USD price range: ${token1_usd_price.min():.6f} - ${token1_usd_price.max():.6f}")
            
            # Show sample results
            print(f"\nSample results:")
            sample_data = hourly_data[['quotePrice', 'token1_usd_price', 'eth_usd_price']].head(10)
            print(sample_data)
            
        else:
            print(f"❌ quotePrice column not found in {pool_name}")
            return None
        
        # Save processed data
        output_file = os.path.join(DATA_DIR, f"{pool_name}_processed_hourly.csv")
        hourly_data.to_csv(output_file)
        print(f"✓ Saved processed data to: {output_file}")
        
        return hourly_data
        
    except Exception as e:
        print(f"❌ Error processing {pool_name}: {e}")
        import traceback
        traceback.print_exc()
        return None

In [15]:
# Process all pools
print("Starting pool processing...")
print("=" * 60)

all_processed_pools = {}
for pool_file in pool_csv_files:
    result = process_pool_with_eth_usd(pool_file, eth_usd_hourly)
    if result is not None:
        pool_name = os.path.basename(pool_file).replace('_bigquery.csv', '')
        all_processed_pools[pool_name] = result
        print(f"✓ Successfully processed: {pool_name}")
    else:
        print(f"❌ Failed to process: {os.path.basename(pool_file)}")
    print()  # Empty line for readability

print("=" * 60)
print("Pool processing complete!")

Starting pool processing...

Processing pool: vita_weth_1
Raw pool data shape: (14466, 22)
Pool data timezone: UTC
Pool data timezone: UTC
Aggregating to hourly...


/Users/nicolaschiavo/Dev/tesi/univ3-strategies/ActiveStrategyFramework.py:204: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  price_data_aggregated = new_data.resample(resample_option).last().copy()


Hourly aggregated shape: (5664, 23)
Hourly data timezone: UTC
Calculating USD prices...
Normal pool calculation
quotePrice = token1/token0, so token1/USD = quotePrice × ETH/USD
USD price calculation complete
Token1 USD price range: $0.635117 - $6.983793

Sample results:
                           quotePrice  token1_usd_price  eth_usd_price
2025-01-01 00:00:00+00:00    0.001709          5.748315        3363.70
2025-01-01 01:00:00+00:00    0.001701          5.692707        3346.54
2025-01-01 02:00:00+00:00    0.001701          5.720043        3362.61
2025-01-01 03:00:00+00:00    0.001688          5.662038        3355.20
2025-01-01 04:00:00+00:00    0.001679          5.610064        3341.14
2025-01-01 05:00:00+00:00    0.001679          5.617233        3345.41
2025-01-01 06:00:00+00:00    0.001648          5.516273        3346.60
2025-01-01 07:00:00+00:00    0.001652          5.530450        3347.12
2025-01-01 08:00:00+00:00    0.001652          5.511269        3337.01
2025-01-01 09:00:00

/Users/nicolaschiavo/Dev/tesi/univ3-strategies/ActiveStrategyFramework.py:204: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  price_data_aggregated = new_data.resample(resample_option).last().copy()


Hourly aggregated shape: (5664, 23)
Hourly data timezone: UTC
Calculating USD prices...
Normal pool calculation
quotePrice = token1/token0, so token1/USD = quotePrice × ETH/USD
USD price calculation complete
Token1 USD price range: $10.166388 - $27.328013

Sample results:
                           quotePrice  token1_usd_price  eth_usd_price
2025-01-01 00:00:00+00:00    0.005998         20.176539        3363.70
2025-01-01 01:00:00+00:00    0.005981         20.016675        3346.54
2025-01-01 02:00:00+00:00    0.005992         20.149102        3362.61
2025-01-01 03:00:00+00:00    0.006001         20.133847        3355.20
2025-01-01 04:00:00+00:00    0.005981         19.983550        3341.14
2025-01-01 05:00:00+00:00    0.005986         20.024400        3345.41
2025-01-01 06:00:00+00:00    0.005986         20.031412        3346.60
2025-01-01 07:00:00+00:00    0.005985         20.032614        3347.12
2025-01-01 08:00:00+00:00    0.005963         19.898531        3337.01
2025-01-01 09:00:

...

Hourly aggregated shape: (5664, 23)
Hourly data timezone: UTC
Calculating USD prices...
Normal pool calculation
quotePrice = token1/token0, so token1/USD = quotePrice × ETH/USD
USD price calculation complete
Token1 USD price range: $0.824893 - $4.122873

Sample results:
                           quotePrice  token1_usd_price  eth_usd_price
2025-01-01 00:00:00+00:00    0.001024          3.443915        3363.70
2025-01-01 01:00:00+00:00    0.001038          3.473769        3346.54
2025-01-01 02:00:00+00:00    0.001061          3.567391        3362.61
2025-01-01 03:00:00+00:00    0.001043          3.500445        3355.20
2025-01-01 04:00:00+00:00    0.001042          3.482332        3341.14
2025-01-01 05:00:00+00:00    0.001033          3.455921        3345.41
2025-01-01 06:00:00+00:00    0.001043          3.491023        3346.60
2025-01-01 07:00:00+00:00    0.001033          3.457184        3347.12
2025-01-01 08:00:00+00:00    0.001033          3.446742        3337.01
2025-01-01 09:00:00

/Users/nicolaschiavo/Dev/tesi/univ3-strategies/ActiveStrategyFramework.py:204: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  price_data_aggregated = new_data.resample(resample_option).last().copy()


Hourly aggregated shape: (5664, 23)
Hourly data timezone: UTC
Calculating USD prices...
Normal pool calculation
quotePrice = token1/token0, so token1/USD = quotePrice × ETH/USD
USD price calculation complete
Token1 USD price range: $2.533958 - $8.962263

Sample results:
                           quotePrice  token1_usd_price  eth_usd_price
2025-01-01 00:00:00+00:00    0.002046          6.883357        3363.70
2025-01-01 01:00:00+00:00    0.002034          6.806063        3346.54
2025-01-01 02:00:00+00:00    0.002026          6.812350        3362.61
2025-01-01 03:00:00+00:00    0.002017          6.766504        3355.20
2025-01-01 04:00:00+00:00    0.002016          6.735109        3341.14
2025-01-01 05:00:00+00:00    0.002016          6.743040        3345.41
2025-01-01 06:00:00+00:00    0.002004          6.706019        3346.60
2025-01-01 07:00:00+00:00    0.002006          6.715655        3347.12
2025-01-01 08:00:00+00:00    0.002007          6.698837        3337.01
2025-01-01 09:00:00

/Users/nicolaschiavo/Dev/tesi/univ3-strategies/ActiveStrategyFramework.py:204: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  price_data_aggregated = new_data.resample(resample_option).last().copy()


Hourly aggregated shape: (5664, 23)
Hourly data timezone: UTC
Calculating USD prices...
INJ pool detected - using inverted calculation
quotePrice = INJ/ETH, so INJ/USD = (1/quotePrice) × ETH/USD
USD price calculation complete
Token1 USD price range: $6.333724 - $26.319296

Sample results:
                           quotePrice  token1_usd_price  eth_usd_price
2025-01-01 00:00:00+00:00  169.534357         19.840816        3363.70
2025-01-01 01:00:00+00:00  169.896657         19.697504        3346.54
2025-01-01 02:00:00+00:00  170.429088         19.730259        3362.61
2025-01-01 03:00:00+00:00  171.922682         19.515750        3355.20
2025-01-01 04:00:00+00:00  172.717218         19.344568        3341.14
2025-01-01 05:00:00+00:00  172.717218         19.369291        3345.41
2025-01-01 06:00:00+00:00  173.003714         19.344093        3346.60
2025-01-01 07:00:00+00:00  172.606876         19.391580        3347.12
2025-01-01 08:00:00+00:00  172.606876         19.333007        3337.01
